In [ ]:
%matplotlib inline

In [ ]:
import autograd.numpy as np
from autograd import grad, jacobian

In [ ]:
import matplotlib.pyplot as plt
from mpltools import annotation

# f & g example

In [ ]:
def f(x):
    return x[0] * x[1]

In [ ]:
def grad_f_expected(x):
    return np.flip(x)

In [ ]:
x = np.array([1.0, 2.0])
print(f"x = {x}")
print(f"f(x) = {f(x)}")

In [ ]:
grad_f = grad(f)
print(f"∇f(x) = {grad_f(x)}")
assert np.allclose(grad_f(x), grad_f_expected(x))

In [ ]:
def g(y):
    return np.array([np.sin(y), np.cos(y)])

In [ ]:
def grad_g_expected(y):
    return np.array([np.cos(y), -np.sin(y)])

In [ ]:
y = np.pi / 4
print(f"y = {y}")
print(f"g(y) = {g(y)}")

In [ ]:
grad_g = jacobian(g)

In [ ]:
print(f"∇g(y) = {grad_g(y)}")
assert np.allclose(grad_g(y), grad_g_expected(y))

### Taylor test

In [ ]:
def h(x):
    return g(f(x))

In [ ]:
# Choose arbitrary inputs for the composition
x = np.array([1.0, 1.0])

# Choose seed vector
xd = np.array([1.0, 0.0])

# Compute the derivative using Autograd
yd = jacobian(h)(x) @ xd

# Run the Taylor test over several spacing values
spacings = [1.0, 0.1, 0.01, 0.001]
errors = []
for spacing in spacings:

    # Compute the perturbation in the seed vector direction
    epsilon = spacing * xd

    # Compute the discrepancy
    errors.append(np.linalg.norm(h(x + epsilon) - h(x) - spacing * yd))

# Plot the solution, demonstrating that the expected quadratic convergence is achieved
fig, axes = plt.subplots()
axes.loglog(spacings, errors, "--x")
axes.set_xlabel(r"$\epsilon$ spacing")
axes.set_ylabel(r"$\ell_2$ error")
annotation.slope_marker((1e-2, 1e-4), 2, ax=axes, invert=True)
axes.grid()

## ODE-constrained optimisation example

In [ ]:
def initial_condition():
    """
    Apply the initial condition for the ODE initial value problem
        du/dt = u, u(0) = 1
    """
    u0 = 1.0
    return u0

In [ ]:
def theta_step(u_, dt, theta):
    """
    Take a single iteration of a theta method for solving the ODE initial value problem
        du/dt = u, u(0) = 1
    """
    u = u_ * (1.0 + dt * (1.0 - theta)) / (1.0 - dt * theta)
    return u

In [ ]:
trajectories = {}

In [ ]:
def theta_method(theta):
    """
    Solve the ODE initial value problem
        du/dt = u, u(0) = 1
    using a theta timestepping method, returning the solution trajectory.
    """
    t = 0.0
    dt = 0.1
    end_time = 1.0
    u0 = initial_condition()

    # Timestepping loop
    trajectory = [u0]
    u_ = u0
    while t < end_time - 1.0e-05:
        u = theta_step(u_, dt, theta)
        u_ = u
        t += dt
        trajectory.append(u)
    trajectories[theta] = trajectory
    return u

In [ ]:
def cost_function(theta):
    """
    Cost function evaluating the l2 error at the end time against the analytical solution u(t)=exp(t)
    """
    u = theta_method(theta)
    e = np.exp(1.0)
    return (u - e) ** 2

In [ ]:
controls = []
costs = []

In [ ]:
def gradient_descent(maxiter=1000, gtol=1.0e-05, dtol=1.1, alpha=0.10):
    """
    Function for optimising the theta parameter for a theta timestepping method for solving the ODE
        du/dt = u, u(0)=1
    using gradient descent.
    """
    # Start from forward Euler
    theta = 0.0

    for i in range(maxiter):
        J = cost_function(theta)
        Jd = grad(cost_function)(theta)
        
        controls.append(theta)
        costs.append(J)
        
        # Convergence and divergence checks
        if i == 0:
            J_init = J
        elif abs(Jd / Jd_) < gtol:
            print(f"Converged in {i+1} iterations due to gradient convergence")
            return theta
        elif abs(J / J_init) > dtol:
            raise RuntimeError(f"Detected divergence after {i+1} iterations")
        Jd_ = Jd

        # Take a step in the descent direction
        p = -Jd
        theta += alpha * p
    # raise RuntimeError("Reached maximum iteratons without convergence")  # FIXME: make more robust
    return theta

In [ ]:
theta_opt = gradient_descent()

In [ ]:
costs[0] = np.nan  # Remove the first entry because it's uninitialised garbage

fig, axes = plt.subplots(ncols=2, figsize=(12, 5))
axes[0].loglog(costs, "--", label="Cost function value")
axes[0].legend()
axes[0].grid()
axes[1].plot(controls, "--", label="Control value")
axes[1].legend()
axes[1].grid()

In [ ]:
theta_method(0.0)
forward = trajectories[0.0]
theta_method(1.0)
backward = trajectories[1.0]
theta_method(theta_opt)
optimised = trajectories[theta_opt]

times = np.linspace(0, 1, len(forward))

fig, axes = plt.subplots()
axes.plot(times, np.exp(times), "-", color="k", label="Analytical solution")
axes.plot(times, forward, "--x", label="Forward Euler")
axes.plot(times, backward, ":o", label="Backward Euler")
axes.plot(times, optimised, "-.^", label=rf"Optimised ($\theta={theta_opt:.4f}$)")
axes.legend()
axes.grid()